In [ ]:
# Import der benötigten Bibliotheken
import pandas as pd
import numpy as np
import re
import spacy
import nltk
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from gensim.models import Word2Vec

In [ ]:
# Einlesen des Datensatzes aus dem data-Ordner
df = pd.read_csv("data/complaints_processed.csv")
print("Original:", len(df))

# leere Textfelder in der Spalte "Narrative" entfernen
df = df.dropna(subset=["narrative"])
print("Nach dropna:", len(df))

# Stichprobe von 10000 Texten ziehen
df = df.sample(10000, random_state=42)
print("Nach sample:", len(df))

# Auswahl der Spalte, die die Beschwerden enthält
texts = df["narrative"]
print("Texts:", len(texts))

In [ ]:
nltk.download('stopwords')

# Laden von spaCy zur Tokenisierung und Lemmatisierung
nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))

# Alle Vorverarbeitungsschritte
def preprocess(text):
    # Kleinbuchstaben
    text = text.lower()

    # Sonderzeichen & Zahlen entfernen
    text = re.sub(r'[^a-z\s]', ' ', text)

    # überflüssige Leerzeichen entfernen
    text = re.sub(r'\s+', ' ', text).strip()

    #Tokenisierung mit spaCy
    doc = nlp(text)

    # Erstellung der Tokenliste: Entfernen kurzer Bewertungen und Stoppwörter und mit weniger als 3 Wörtern und Lemmatisierung
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in stop_words and len(token.lemma_) > 2
    ]
    # Rückgabe der gefilterten Tokenliste
    return tokens

# anwenden der Vorverarbeitung
processed_tokens = texts.apply(preprocess)


# zurück in String umwandeln
clean_texts = processed_tokens.apply(lambda x: " ".join(x))

print(clean_texts.head())
print("Anzahl gültiger Texte:", len(clean_texts))

In [ ]:
# TF-IDF-Vektorisierung mit maximal 3000 Wörtern als Features
tfidf = TfidfVectorizer(max_features=3000)
tfidf_matrix = tfidf.fit_transform(clean_texts)

print("TF-IDF Matrix:", tfidf_matrix.shape)

In [ ]:
# Vorbereitung Word2Vec
tokenized = [text.split() for text in clean_texts]

#Word2Vec-Vektorisierung
w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=5
)
# Erzeugung der Vektoren
def document_vector(tokens):
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)
w2v_vectors = np.array([document_vector(text) for text in tokenized])

print("Word2Vec Matrix:", w2v_vectors.shape)

In [ ]:
#Themenextraktion mit LDA
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(tfidf_matrix)

def print_topics(model, feature_names, n_words=10):
    for i, topic in enumerate(model.components_):
        words = [feature_names[j] for j in topic.argsort()[-n_words:]]
        print(f"\nTopic {i+1}:")
        print(", ".join(words))

print("LDA Topics:")
print_topics(lda, tfidf.get_feature_names_out())

In [ ]:
# Zhemenextraktion mit NMF
nmf = NMF(n_components=5, random_state=42)
nmf.fit(tfidf_matrix)

print("NMF Topics:")
print_topics(nmf, tfidf.get_feature_names_out())